# Silver: ERP Customers
**Source:** databricks_bootcamp_dwb.bronze.erp_px_cat_g1v2  
**Target:** databricks_bootcamp_dwb.silver.erp_products

ERP = 'Enterprise Resource Planning'

# Read Data From Bronze Layer

In [0]:
# imports
from pyspark.sql.functions import count, when, isnull, col, trim
from pyspark.sql.types import StringType, IntegerType, DateType

In [0]:
df = spark.table("databricks_bootcamp_dwb.bronze.erp_px_cat_g1v2")
df.display()

# EDA

In [0]:
# Check for nulls in each column
df.select([count(when(isnull(c), c)).alias(c) for c in df.columns]).display()

In [0]:
# Check for duplicates in each column
print(f"total rows: {df.count()}")
for column in df.columns:
    duplicate_count = df.groupBy(column).count().filter(col("count") > 1).agg(count("*")).collect()[0][0]
    print(f"{column}: {duplicate_count} duplicate values")

In [0]:
# Get distinct values from categorical variables.
cat_columns=["CAT", "MAINTENANCE", "SUBCAT"]

for column in cat_columns:
    print(f"Distinct values in {column}:")
    df.select(column).distinct().display()

In [0]:
# Check data types of all columns
df.printSchema()

# Data Transformations

In [0]:
# trim strings
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

display(df)

In [0]:
# make friendly column names
RENAME_MAP = {
    "CAT": "category",
    "SUBCAT": "sub_category",
    "MAINTENANCE": "maintenance"
}

df = df.select([col(c).alias(RENAME_MAP.get(c, c)) for c in df.columns])
display(df)

In [0]:
# Schema transformation for join compatibility
# Justification: see table_relationship_exploration notebook and silver_crm_prd_info for pattern
# Transform ID: replace underscores with hyphens, rename to category_id
# Then drop ID column to avoid duplication

from pyspark.sql.functions import regexp_replace

# Replace underscores with hyphens in ID and rename to category_id
df = df.withColumn("category_id", regexp_replace(col("ID"), "_", "-"))

# Drop ID column
df = df.drop("ID")

print("Schema after transformation:")
df.printSchema()
display(df)

In [0]:
# Define target data types for non-string columns.
type_mappings = {
}

# Apply type casting
for field in df.schema.fields:
    column_name = field.name
    
    if column_name in type_mappings:
        target_type = type_mappings[column_name]
        if target_type is not None:
            df = df.withColumn(column_name, col(column_name).cast(target_type))
    else:
        # All other columns should be StringType
        df = df.withColumn(column_name, col(column_name).cast(StringType()))

print("Data types after enforcement:")
df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp

# Add timestamp column to track when row was written to silver table
df = df.withColumn("silver_updated_at", current_timestamp())

print(f"Added silver_updated_at column")
display(df)

# Write To Silver Table

In [0]:
df.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("databricks_bootcamp_dwb.silver.erp_products")

In [0]:
%sql
SELECT * FROM databricks_bootcamp_dwb.silver.erp_products
LIMIT 100